# 11 -- Data Validation and Item Lookup: the `vdl data` Subapp

Before plotting anything, you want to know: *is the input file healthy,*
*how big are the sets, what's the right model, and where does my favourite*
*gene live?* The `vdl data` subapp answers all of these in script-friendly
form (JSON / plain text), so a CI workflow or pipeline can decide what to
do next without an interactive prompt.

**Audience:** bioinformaticians who validate inputs and look up specific
genes across multiple datasets.

**What you will learn:**

- `vdl data validate` for schema + content checks (JSON + `--strict`)
- `vdl data describe` for a quick text summary
- `vdl data lookup` to find which Venn region a gene lives in
- `vdl data fit-model` for model recommendations
- `vdl data convert` for TSV<->CSV format conversion
- A typical data-hygiene checklist for new inputs


In [1]:
import json
import subprocess
import sys
from pathlib import Path

VDL = str(Path(sys.executable).parent / 'vdl')
print('Using vdl:', VDL)

Using vdl: /Users/Zoli/anaconda3/bin/vdl


## 1. `vdl data validate` -- structured JSON report

By default `vdl data validate` emits a JSON document with `sets`,
`item_count`, `errors`, `warnings`, `info`, and `exit_code` keys. The
command exits non-zero iff `errors` is non-empty.


In [2]:
r = subprocess.run(
    [VDL, 'data', 'validate', '--sample'],
    capture_output=True, text=True, check=True,
)
report = json.loads(r.stdout)
print('Input        :', report['input'])
print('Sets         :', report['sets'])
print('Item count   :', report['item_count'])
print('Errors       :', report['errors'])
print('Warnings     :', report['warnings'])
print('Exit code    :', report['exit_code'])
print()
print('Per-set sizes (from info field):')
for entry in report['info']:
    if entry.get('kind') == 'set-size':
        print(f'  {entry["set"]:20s}  {entry["count"]:>6,} items')

Input        : dataset_real_cancer_drivers_4
Sets         : ['Vogelstein', 'COSMIC_CGC', 'OncoKB', 'IntOGen']
Item count   : 1394
Errors       : []
Warnings     : []
Exit code    : 0

Per-set sizes (from info field):
  Vogelstein               138 items
  COSMIC_CGC               581 items
  OncoKB                 1,231 items
  IntOGen                  633 items


## 2. `--strict` mode for CI pipelines

With `--strict`, warnings are promoted to errors. This is the right mode
for CI: a non-zero exit code stops the pipeline before a malformed input
reaches the renderer. For the bundled clean sample there are no warnings,
so the strict run also succeeds.


In [3]:
r = subprocess.run(
    [VDL, 'data', 'validate', '--sample', '--strict'],
    capture_output=True, text=True,
)
print(f'Exit code: {r.returncode}')
report = json.loads(r.stdout)
print(f'Errors after strict promotion: {len(report["errors"])}')
print(f'Exit-code field in report   : {report["exit_code"]}')

Exit code: 0
Errors after strict promotion: 0
Exit-code field in report   : 0


## 3. `vdl data lookup` -- find a known gene

`vdl data lookup --sample TP53` walks every Venn region and prints the
ones that contain the requested item. TP53 is in all four cancer-driver
catalogs, so it should appear in the 4-way intersection.


In [4]:
r = subprocess.run(
    [VDL, 'data', 'lookup', '--sample', 'TP53'],
    capture_output=True, text=True, check=True,
)
print(r.stdout)

'TP53' found in 1 region(s):
  region ABCD       (4-way: Vogelstein, COSMIC_CGC, OncoKB, IntOGen) — 120 items total



## 4. `vdl data lookup` -- gene not in the dataset

If the item is not in any set, `lookup` prints a one-line *not found*
message and exits cleanly (exit code 0). Scripts can detect this by
matching the string `'not found'` in stdout.


In [5]:
r = subprocess.run(
    [VDL, 'data', 'lookup', '--sample', 'NOTAGENE'],
    capture_output=True, text=True, check=True,
)
print(r.stdout)
print('Found?', 'not found' not in r.stdout)

'NOTAGENE': not found in dataset universe

Found? False


## 5. Batch lookup: tabulate where each gene lives

Loop over a list of well-known oncogenes and print a compact summary
of which sets each appears in.


In [6]:
genes = ['TP53', 'KRAS', 'MYC', 'BRCA1', 'EGFR']
print(f'{"gene":<8}  {"region(s) hit":<60}')
print('-' * 70)
for g in genes:
    r = subprocess.run(
        [VDL, 'data', 'lookup', '--sample', g],
        capture_output=True, text=True, check=True,
    )
    # First line is the summary; subsequent lines list each region.
    lines = r.stdout.strip().splitlines()
    if 'not found' in lines[0]:
        print(f'{g:<8}  (not in dataset)')
    else:
        print(f'{g:<8}  {lines[0]}')
        for region_line in lines[1:]:
            print(f'          {region_line.strip()}')

gene      region(s) hit                                               
----------------------------------------------------------------------


TP53      'TP53' found in 1 region(s):
          region ABCD       (4-way: Vogelstein, COSMIC_CGC, OncoKB, IntOGen) — 120 items total


KRAS      'KRAS' found in 1 region(s):
          region ABCD       (4-way: Vogelstein, COSMIC_CGC, OncoKB, IntOGen) — 120 items total


MYC       'MYC' found in 1 region(s):
          region ABCD       (4-way: Vogelstein, COSMIC_CGC, OncoKB, IntOGen) — 120 items total


BRCA1     'BRCA1' found in 1 region(s):
          region ABCD       (4-way: Vogelstein, COSMIC_CGC, OncoKB, IntOGen) — 120 items total


EGFR      'EGFR' found in 1 region(s):
          region ABCD       (4-way: Vogelstein, COSMIC_CGC, OncoKB, IntOGen) — 120 items total


## 6. `vdl data describe` -- quick text summary

`describe` is the fastest way to see set names + sizes + a model
recommendation without parsing JSON. Useful at the top of any script.


In [7]:
r = subprocess.run(
    [VDL, 'data', 'describe', '--sample'],
    capture_output=True, text=True, check=True,
)
print(r.stdout)

sets:   ['Vogelstein', 'COSMIC_CGC', 'OncoKB', 'IntOGen']
items:  1394
model:  auto
top regions by exclusive count:
  C           559
  BCD         268
  BC          187
  D           156
  ABCD        120



## 7. `vdl data fit-model` -- which Venn template fits?

`fit-model` looks at the set count and suggests a model name, plus the
list of all bundled candidates for that count. The default suggestion is
what `analyze(model='auto')` would pick.


In [8]:
r = subprocess.run(
    [VDL, 'data', 'fit-model', '--sample'],
    capture_output=True, text=True, check=True,
)
print(r.stdout)

suggested model: venn-4e-set-euler
available models for N=4: ['venn-4-set', 'venn-4a-set-edwards', 'venn-4b-set-anderson', 'venn-4e-set-carroll-triangle', 'venn-4e-set-euler', 'venn-4e-set-rectangle', 'venn-4f-set']



## 8. `vdl data convert` -- TSV <-> CSV roundtrip

Some downstream tools insist on CSV; others on TSV. `vdl data convert`
swaps the delimiter while preserving the binary-matrix structure.
It takes two explicit file-path arguments (`input output`) and infers the
format from each extension. Below we (a) write the sample's region-summary
TSV via `vdl export`, (b) convert it to CSV, (c) round-trip back to TSV,
and (d) verify the row count is preserved.


In [9]:
import tempfile

tmp = Path(tempfile.mkdtemp(prefix='vdl_convert_'))

# Step 1: get a real TSV on disk from the bundled sample.
tsv_in = tmp / 'sample.tsv'
subprocess.run(
    [VDL, 'export', 'region-summary', '--sample', '--out', str(tsv_in)],
    check=True,
)
print(f'TSV (input)        : {tsv_in} ({tsv_in.stat().st_size:,} bytes)')

# Step 2: TSV -> CSV.
csv_out = tmp / 'sample.csv'
subprocess.run(
    [VDL, 'data', 'convert', str(tsv_in), str(csv_out)],
    check=True,
)
print(f'CSV                 : {csv_out} ({csv_out.stat().st_size:,} bytes)')

# Step 3: CSV -> TSV (round-trip).
tsv_out = tmp / 'sample_roundtrip.tsv'
subprocess.run(
    [VDL, 'data', 'convert', str(csv_out), str(tsv_out)],
    check=True,
)
print(f'TSV (round-tripped) : {tsv_out} ({tsv_out.stat().st_size:,} bytes)')

# Step 4: spot-check row counts.
in_rows = sum(1 for _ in tsv_in.open())
csv_rows = sum(1 for _ in csv_out.open())
tsv_rows = sum(1 for _ in tsv_out.open())
print(f'Rows: input={in_rows}, csv={csv_rows}, roundtrip={tsv_rows}, '
      f'match={in_rows == csv_rows == tsv_rows}')

note: --sample → using bundled 'dataset_real_cancer_drivers_4'


Wrote /var/folders/ww/fsvzxkwx6hgfzqbq08lfrqdc0000gn/T/vdl_convert_fgaj2igx/sample.tsv
TSV (input)        : /var/folders/ww/fsvzxkwx6hgfzqbq08lfrqdc0000gn/T/vdl_convert_fgaj2igx/sample.tsv (8,897 bytes)


Wrote /var/folders/ww/fsvzxkwx6hgfzqbq08lfrqdc0000gn/T/vdl_convert_fgaj2igx/sample.csv
CSV                 : /var/folders/ww/fsvzxkwx6hgfzqbq08lfrqdc0000gn/T/vdl_convert_fgaj2igx/sample.csv (8,898 bytes)


Wrote /var/folders/ww/fsvzxkwx6hgfzqbq08lfrqdc0000gn/T/vdl_convert_fgaj2igx/sample_roundtrip.tsv
TSV (round-tripped) : /var/folders/ww/fsvzxkwx6hgfzqbq08lfrqdc0000gn/T/vdl_convert_fgaj2igx/sample_roundtrip.tsv (8,898 bytes)
Rows: input=16, csv=16, roundtrip=16, match=True


## Typical data-hygiene checklist

Before running a full pipeline on a new input, walk through:

1. **`vdl data validate <input> --strict`** -- schema clean? Fail-fast if not.
2. **`vdl data describe <input>`** -- set count + sizes look plausible?
3. **`vdl data fit-model <input>`** -- pick a model (or stick with auto).
4. **`vdl data lookup <input> <known-gene>`** -- sanity-check at least one
   expected gene appears where you think it should.
5. **`vdl data convert <input> --to ...`** -- only if a downstream tool
   requires a specific delimiter.

Then proceed to `vdl analyze` or `vdl workflow run-from` with confidence.


## Next steps

- [`09_cli_workflows.ipynb`](09_cli_workflows.ipynb) -- broader CLI walkthrough including `vdl workflow run-from`
- [`06_pipeline_integration.ipynb`](06_pipeline_integration.ipynb) -- wire validation + analysis into Snakemake or Nextflow
